# Ariel 2025 — Notebook 2: Experiments

Chạy toàn bộ thí nghiệm trên features đã build (Notebook 1):

| Exp | Nội dung |
|---|---|
| 0 | Sanity check / EDA |
| 1 | Baseline vật lý đơn giản (Ridge trên depth features) |
| 2 | **So sánh 7 họ mô hình** (benchmark, official GLL) |
| 3 | Target PCA — quét `n_components` |
| 4 | **Ablation PHC** (scalar → per-wavelength → feature-conditioned) — đóng góp chủ chốt |
| 5 | Neural baseline + ghi chú deep sequence models |
| 6 | **GLL-weighted family mixture** (PHC bước 3) |
| 7 | Hyperparameter search + refit + submission |

In [ ]:
# === Setup: make the ariel_ml package importable ===
# The ariel_ml source must be available in the Kaggle session. Pick ONE:
#   (a) Add the repo as a Kaggle Dataset and point CANDIDATE_SRC at its /src.
#   (b) Clone it:  !git clone https://github.com/<you>/ML_IT3190E_Project /kaggle/working/ML_IT3190E_Project
#   (c) pip install from GitHub:  !pip install -q git+https://github.com/<you>/ML_IT3190E_Project
import sys
from pathlib import Path

CANDIDATE_SRC = [
    "/kaggle/input/ariel-ml-src/src",            # repo uploaded as a Kaggle dataset
    "/kaggle/working/ML_IT3190E_Project/src",    # repo cloned into working dir
    "/kaggle/usr/lib/ariel_ml",
    "src",                                        # running locally from repo root
    "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p)
        print("Using ariel_ml from:", _p)
        break
else:
    print("WARNING: ariel_ml source not found. Attach/clone the repo (see comments above).")

# Optional model dependencies (LightGBM/XGBoost are preinstalled on Kaggle; ngboost is not):
# !pip install -q ngboost

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## Cấu hình chung

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import replace
import matplotlib.pyplot as plt

from ariel_ml.config import ModelConfig
from ariel_ml.dataset_builder import align_features_and_targets
from ariel_ml.benchmark import benchmark_models
from ariel_ml.training import (
    cross_validate_model, hyperparameter_search, refit_full_model, build_gll_weighted_ensemble,
)
from ariel_ml.metrics import ariel_naive_reference, ariel_gll_score

# Đường dẫn features (đổi nếu attach từ dataset khác)
FEATURES_TRAIN = OUTPUT_DIR / "features_train.csv"
if not FEATURES_TRAIN.exists():
    for c in ["/kaggle/input/ariel-features/features_train.csv", "outputs/features_train.csv"]:
        if Path(c).exists():
            FEATURES_TRAIN = Path(c); break
FEATURES_TEST = OUTPUT_DIR / "features_test.csv"

N_COMPONENTS = 30
N_SPLITS = 5
SIGMA_CAL_FRACTION = 0.2   # giữ 20% train fold cho sigma calibration -> tránh circular NLL
RANDOM_STATE = 42
USE_GPU = False   # chỉ tăng tốc lightgbm/xgboost (lightgbm cần build GPU); model khác vẫn CPU
print("Features train:", FEATURES_TRAIN, FEATURES_TRAIN.exists())


In [ ]:
features = pd.read_csv(FEATURES_TRAIN)
targets = pd.read_csv(DATA_ROOT / "train.csv")
X, y, groups, target_columns = align_features_and_targets(features, targets)
Xv = X.to_numpy(dtype=float)
print("X:", Xv.shape, "| y:", y.shape, "| n_planets:", len(np.unique(groups)),
      "| n_features:", Xv.shape[1], "| n_targets:", y.shape[1])


## Exp 0 — Sanity check / EDA

In [ ]:
print("Target (transit depth) stats:")
print("  mean:", float(np.mean(y)), " std:", float(np.std(y)),
      " min:", float(np.min(y)), " max:", float(np.max(y)))
print("NaN in features:", int(np.isnan(Xv).sum()), " | NaN in y:", int(np.isnan(y).sum()))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i in range(min(5, y.shape[0])):
    ax[0].plot(y[i], lw=1)
ax[0].set_title("Sample target spectra"); ax[0].set_xlabel("wavelength index"); ax[0].set_ylabel("(Rp/Rs)^2")
ax[1].plot(np.std(y, axis=0)); ax[1].set_title("Per-wavelength std of targets"); ax[1].set_xlabel("wavelength index")
plt.tight_layout(); plt.show()


## Exp 1 — Baseline vật lý đơn giản
Ridge chỉ dùng nhóm **depth features** (transit depth ≈ `(Rp/Rs)^2`), so với Ridge dùng toàn bộ feature.

In [ ]:
depth_cols = [i for i, c in enumerate(X.columns) if "depth" in c.lower()]
print("Depth features:", len(depth_cols), "/", Xv.shape[1])

cfg1 = ModelConfig(n_components=min(10, N_COMPONENTS), random_state=RANDOM_STATE)
def cv_gll(xmat, model_name, cfg):
    m = cross_validate_model(xmat, y, model_name=model_name, model_config=cfg,
                             n_splits=N_SPLITS, groups=groups, random_state=RANDOM_STATE,
                             sigma_cal_fraction=SIGMA_CAL_FRACTION).mean_metrics
    return m

m_depth = cv_gll(Xv[:, depth_cols] if depth_cols else Xv, "ridge", cfg1)
m_full = cv_gll(Xv, "ridge", ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE))
print(f"Ridge depth-only : GLL={m_depth['ariel_gll_score']:.4f}  RMSE={m_depth['rmse_mean']:.3e}")
print(f"Ridge full feats : GLL={m_full['ariel_gll_score']:.4f}  RMSE={m_full['rmse_mean']:.3e}")


## Exp 2 — So sánh 7 họ mô hình (benchmark harness)
Cùng GroupKFold folds, cùng `n_components`, chấm bằng metric Ariel chính thức.
Model thiếu optional dep được báo `skipped` (không lỗi).

In [ ]:
MODEL_NAMES = [
    "ridge", "lasso", "elastic_net",                    # linear
    "bayesian_ridge", "ard",                            # bayesian
    "svr", "kernel_ridge",                              # kernel / svm
    "knn",                                              # neighbors
    "random_forest", "extra_trees", "boosting", "hist_gradient_boosting",  # trees
    "mlp",                                              # neural
    "lightgbm", "xgboost",                              # optional (skip nếu thiếu)
]
# gaussian_process / ngboost rất chậm trên nhiều planet — thêm nếu cần:
# MODEL_NAMES += ["gaussian_process", "ngboost"]

bench_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU)
result = benchmark_models(Xv, y, model_names=MODEL_NAMES, model_config=bench_cfg,
                          n_splits=N_SPLITS, groups=groups, random_state=RANDOM_STATE,
                          sigma_cal_fraction=SIGMA_CAL_FRACTION)
table = result.to_frame()
table.to_csv(OUTPUT_DIR / "exp2_benchmark.csv", index=False)
best = result.best("ariel_gll_score", maximize=True)
print("Best by Ariel GLL:", best.model_name, "(", best.family, ") =", round(best.metrics["ariel_gll_score"], 4))
table


In [ ]:
ok = table[table["status"] == "ok"].sort_values("ariel_gll_score")
plt.figure(figsize=(9, max(3, 0.4 * len(ok))))
plt.barh(ok["model"], ok["ariel_gll_score"], color="steelblue")
plt.xlabel("Ariel GLL score (higher = better)"); plt.title("Exp 2 — model family comparison")
plt.tight_layout(); plt.show()


## Exp 3 — Target PCA: quét `n_components`

In [ ]:
grid = [10, 20, 30, 40, 50]
rows = []
for nc in grid:
    m = cv_gll(Xv, "bayesian_ridge", ModelConfig(n_components=nc, random_state=RANDOM_STATE))
    rows.append({"n_components": nc, "ariel_gll_score": m["ariel_gll_score"], "rmse_mean": m["rmse_mean"]})
pca_table = pd.DataFrame(rows)
pca_table.to_csv(OUTPUT_DIR / "exp3_pca.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(pca_table["n_components"], pca_table["ariel_gll_score"], "o-", color="tab:blue", label="GLL")
ax1.set_xlabel("n_components"); ax1.set_ylabel("Ariel GLL", color="tab:blue")
ax2 = ax1.twinx(); ax2.plot(pca_table["n_components"], pca_table["rmse_mean"], "s--", color="tab:red", label="RMSE")
ax2.set_ylabel("RMSE", color="tab:red"); plt.title("Exp 3 — Target PCA sweep"); plt.tight_layout(); plt.show()
pca_table


## Exp 4 — Ablation PHC (đóng góp chủ chốt)
So sánh các chế độ hiệu chỉnh σ: **scalar toàn cục → per-wavelength → feature-conditioned**.
Tất cả dùng cùng model & folds; chỉ khác tầng calibration.

In [ ]:
PHC_MODELS = ["bayesian_ridge", "random_forest"]
base = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, calibrate_sigma=True)
modes = {
    "scalar":            replace(base),
    "per_wavelength":    replace(base, sigma_per_target=True),
    "feature_conditioned": replace(base, sigma_feature_conditioned=True),
}
rows = []
for model_name in PHC_MODELS:
    for label, cfg in modes.items():
        m = cv_gll(Xv, model_name, cfg)
        rows.append({"model": model_name, "calibration": label,
                     "ariel_gll_score": m["ariel_gll_score"],
                     "gaussian_nll": m["gaussian_nll"],
                     "coverage_1sigma": m["coverage_1sigma"]})
phc_table = pd.DataFrame(rows)
phc_table.to_csv(OUTPUT_DIR / "exp4_phc_ablation.csv", index=False)
phc_table


In [ ]:
pivot = phc_table.pivot(index="model", columns="calibration", values="ariel_gll_score")
pivot = pivot[["scalar", "per_wavelength", "feature_conditioned"]]
pivot.plot(kind="bar", figsize=(8, 4))
plt.ylabel("Ariel GLL score"); plt.title("Exp 4 — PHC calibration ablation"); plt.xticks(rotation=0)
plt.legend(title="calibration"); plt.tight_layout(); plt.show()
pivot


### Reliability / coverage
Coverage thực nghiệm trong ±1σ / ±2σ nên gần 68% / 95% nếu σ được calibrate tốt.

In [ ]:
for model_name in PHC_MODELS:
    sub = phc_table[phc_table["model"] == model_name]
    print(model_name)
    print(sub[["calibration", "coverage_1sigma", "ariel_gll_score"]].to_string(index=False))
    print()


## Exp 5 — Neural baseline
`mlp` (sklearn) trên features dạng bảng làm cầu nối tới deep learning.
> **Lưu ý:** các model deep sequence (CNN1D/TCN/LSTM/GRU/Transformer trong `ariel_ml.deep_models`) nhận input `[samples, time, channels]` từ **light curve thô**, không phải từ features CSV — cần một pipeline tách riêng để dựng tensor chuỗi. Ở đây dùng MLP làm neural baseline so sánh trực tiếp.

In [ ]:
m_mlp = cv_gll(Xv, "mlp", ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE))
print(f"MLP: GLL={m_mlp['ariel_gll_score']:.4f}  RMSE={m_mlp['rmse_mean']:.3e}  NLL={m_mlp['gaussian_nll']:.3f}")


## Exp 6 — GLL-weighted family mixture (PHC bước 3)
Trộn các họ thành hỗn hợp Gauss, trọng số = softmax(validation GLL).

In [ ]:
ens = build_gll_weighted_ensemble(
    Xv, y,
    model_names=("bayesian_ridge", "random_forest", "svr", "kernel_ridge"),
    model_config=ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True),
    validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE,
)
for name, w, s in zip(ens.model_names, ens.weights, ens.val_scores):
    print(f"  {name:16s} weight={w:.3f}  val_GLL={s:.4f}")
print("Mixture dominated by:", ens.model_names[int(np.argmax(ens.weights))])


## Exp 7 — Hyperparameter search + refit + submission
Search chọn theo **`ariel_gll_score`** (tự maximize), refit trên toàn bộ train, rồi tạo submission từ test features.

In [ ]:
search = hyperparameter_search(
    Xv, y,
    model_names=["bayesian_ridge", "ridge", "random_forest", "kernel_ridge"],
    n_components_grid=[20, 30, 40],
    base_config=ModelConfig(random_state=RANDOM_STATE, sigma_per_target=True),
    n_splits=N_SPLITS, groups=groups, random_state=RANDOM_STATE,
    selection_metric="ariel_gll_score", sigma_cal_fraction=SIGMA_CAL_FRACTION,
)
bestc = search.best_candidate
print("Best:", bestc.model_name, "n_components=", bestc.model_config.n_components,
      "GLL=", round(bestc.mean_metrics["ariel_gll_score"], 4))
final_model = refit_full_model(Xv, y, model_name=bestc.model_name, model_config=bestc.model_config)
print("Refit done on", Xv.shape[0], "planets.")


In [ ]:
from ariel_ml.submission import infer_submission_schema, predict_submission, save_submission

if FEATURES_TEST.exists():
    test_features = pd.read_csv(FEATURES_TEST)
    sample = pd.read_csv(DATA_ROOT / "sample_submission.csv")
    schema = infer_submission_schema(sample_submission=sample)
    # Đảm bảo test có đủ feature columns đã dùng khi train
    for col in X.columns:
        if col not in test_features.columns:
            test_features[col] = 0.0
    submission = predict_submission(final_model, test_features,
                                    feature_columns=list(X.columns), schema=schema)
    save_submission(submission, OUTPUT_DIR / "submission.csv")
    print("Saved submission:", submission.shape, "->", OUTPUT_DIR / "submission.csv")
    display(submission.head())
else:
    print("FEATURES_TEST not found — build test features in Notebook 1 (BUILD_TEST=True) first.")


## Tổng kết
Các bảng kết quả đã lưu: `exp2_benchmark.csv`, `exp3_pca.csv`, `exp4_phc_ablation.csv`, và `submission.csv`.
Dùng các bảng/biểu đồ này trực tiếp cho báo cáo (xem `plans/bao_cao_tom_tat.md`).